In [77]:
# Wczytanie danych i ustalenie rodzajów zmiennych
import pandas as pd
import numpy as np
data_metro = pd.read_csv("dane/interpreter.csv", sep="|", encoding="utf-8")
data_metro.columns = [c.lstrip("@") for c in data_metro.columns]
data_waw = pd.read_csv("dane/warszawa_lokale.csv")

data_metro.info()
data_waw.info()

<class 'pandas.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   type         38 non-null     str    
 1   id           38 non-null     int64  
 2   name         38 non-null     str    
 3   railway:ref  38 non-null     str    
 4   colour       35 non-null     str    
 5   start_date   37 non-null     str    
 6   lat          38 non-null     float64
 7   lon          38 non-null     float64
dtypes: float64(2), int64(1), str(5)
memory usage: 2.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 229727 entries, 0 to 229726
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   name             229727 non-null  str    
 1   latitude         229727 non-null  float64
 2   longitude        229727 non-null  float64
 3   city             229727 non-null  str    
 4   street           229727 non-null  str    
 5

In [78]:
print(data_metro[['name', 'lat', 'lon']])
print(data_waw[['name','latitude','longitude']])

                      name        lat        lon
0           Ratusz-Arsenał  52.245216  21.000882
1                 Marymont  52.271577  20.971940
2            Stare Bielany  52.281828  20.949351
3               Wawrzyszew  52.286347  20.939515
4                  Młociny  52.290770  20.929868
5                Słodowiec  52.276826  20.960126
6         Dworzec Wileński  52.253777  21.035797
7         Stadion Narodowy  52.246835  21.042847
8   Centrum Nauki Kopernik  52.239915  21.031788
9   Nowy Świat-Uniwersytet  52.236820  21.016817
10               Rondo ONZ  52.233074  20.998102
11      Rondo Daszyńskiego  52.230083  20.982895
12                  Kabaty  52.132076  21.065071
13                Stokłosy  52.156076  21.034723
14                 Natolin  52.141101  21.056435
15                 Imielin  52.149300  21.046106
16                 Ursynów  52.162046  21.027628
17                  Służew  52.172762  21.026287
18              Racławicka  52.198864  21.012235
19                Wi

In [79]:
lat_m = data_metro["lat"].to_numpy()[:, None]   # kolumna
lon_m = data_metro["lon"].to_numpy()[:, None]
lat_s = data_waw["latitude"].to_numpy()[None, :]       # wiersz
lon_s = data_waw["longitude"].to_numpy()[None, :]

K = 111_320
dy = (lat_m - lat_s) * K
dx = (lon_m - lon_s) * K * np.cos(np.radians(lat_m))

print(lat_m)
print(lat_s)

D = np.hypot(dx, dy)
assert D.shape == (len(data_waw), len(data_metro)), D.shape
idx = D.argmin(axis=1)

data_waw["metro"] = data_metro["name"].to_numpy()[idx]
data_waw["metro_dist_m"] = D.min(axis=1).round(1)

[[52.2452163]
 [52.2715768]
 [52.2818277]
 [52.2863474]
 [52.2907703]
 [52.2768261]
 [52.2537771]
 [52.2468346]
 [52.2399148]
 [52.2368196]
 [52.2330735]
 [52.2300827]
 [52.1320765]
 [52.1560759]
 [52.1411007]
 [52.1493   ]
 [52.1620456]
 [52.1727624]
 [52.1988637]
 [52.1898719]
 [52.1818168]
 [52.2087775]
 [52.2186581]
 [52.2310069]
 [52.2580586]
 [52.2692619]
 [52.2324542]
 [52.2751021]
 [52.2350954]
 [52.2391822]
 [52.2376624]
 [52.2634709]
 [52.2692518]
 [52.2392071]
 [52.2403314]
 [52.293585 ]
 [52.2920848]
 [52.2837496]]
[[52.2296756  52.20605147 52.20605147 ... 52.2802516  52.22835763
  52.24943115]]


AssertionError: (38, 229727)

In [86]:
m = data_metro[["lon", "lat"]].to_numpy()   # stacje metra
l = data_waw[["longitude", "latitude"]].to_numpy()     # lokale

K = 111_320   # metrów na stopień szerokości

def najblizsza_stacja(l, m):
    roznice = (l[:, None, :] - m[None, :, :]) * K
    roznice[:, :, 0] *= np.cos(np.radians(l[:, None, 1]))   # korekta dla długości
    kwadraty = (roznice ** 2).sum(axis=2)
    return kwadraty.argmin(axis=1), np.sqrt(kwadraty.min(axis=1)).round(0)

idx, odleglosci = najblizsza_stacja(l, m)
data_waw["metro"] = data_metro["name"].to_numpy()[idx]
data_waw["metro_dist_m"] = odleglosci
data_waw[["metro", "metro_dist_m"]].head(10)

,metro,metro_dist_m
0,Centrum,203.0
1,Pole Mokotowskie,2130.0
2,Pole Mokotowskie,2130.0
3,Pole Mokotowskie,2130.0
4,Pole Mokotowskie,2130.0
5,Pole Mokotowskie,2130.0
6,Pole Mokotowskie,2130.0
7,Pole Mokotowskie,2130.0
8,Stare Bielany,640.0
9,Stare Bielany,640.0


In [87]:
from pathlib import Path

out = Path("dane") / "warszawa_lokale_metro.csv"
tmp = out.with_suffix(".csv.tmp")

data_waw.to_csv(tmp, index=False, encoding="utf-8")
tmp.replace(out)

print(out.resolve(), f"| {len(data_waw)} wierszy")

/Users/michal/Documents/VS Code Projects/git_projects/Badanie-Kataster/dane/warszawa_lokale_metro.csv | 229727 wierszy
